# SGB — Sistema de Gestión Bibliotecaria
**Trabajo Final de Semestre · Programación Orientada a Objetos — Ingeniería de Software**

Software en Python (Google Colab) para gestionar una biblioteca: usuarios, libros y categorías,
préstamos y devoluciones, reservas, multas, historial de usuario y análisis de datos.

**Arquitectura de 3 capas** (RNF-01):

| Capa | Sección | Contenido |
|------|---------|-----------|
| Modelo (dominio) | 1 | Enums y funciones `crear_*`, `*_to_linea`, `*_from_linea` de cada entidad |
| Repositorios (persistencia) | 2 | ArchivoUtil + funciones `repo_<entidad>_*` sobre archivos `.txt` |
| Servicios (lógica) | 3 | Funciones `<servicio>_*` con las reglas de negocio y el análisis de datos |
| Interfaz | 4 | Menús con `print()` e `input()` |

> **Decisión de diseño:** el sistema se implementa **solo con funciones**. Cada entidad del dominio
> es un diccionario y cada "clase" del diseño se convierte en un **módulo de funciones** con un prefijo
> común (por ejemplo `UsuarioRepository.guardar()` → `repo_usuario_guardar()`,
> `PrestamoService.prestar()` → `prestamo_prestar()`). Así se conserva la separación en capas sin usar clases ni objetos.

**Cómo usar:** *Entorno de ejecución → Ejecutar todo* y luego interactuar con el menú de la **Celda 5.1**.

Solo se usa la librería estándar de Python 3.10+ (RNF-04).

---
## Celda 0 — Configuración: Drive y rutas
Monta Google Drive y define las rutas de los archivos de datos. Si el notebook se ejecuta fuera de
Colab (por ejemplo en VS Code), usa la carpeta local `SGB/datos/`.

In [ ]:
import os
import re
from datetime import date, datetime, timedelta
from collections import Counter

try:
    from google.colab import drive
    drive.mount('/content/drive')
    RUTA_BASE = '/content/drive/MyDrive/SGB/datos/'
except ImportError:
    # Ejecución local (fuera de Colab)
    RUTA_BASE = os.path.join(os.getcwd(), 'SGB', 'datos') + os.sep

SEPARADOR = '|'          # separador de campos en los .txt
VACIO = 'NINGUNO'        # valor usado para campos opcionales vacíos
MULTA_POR_DIA = 1000     # COP por cada día de retraso
FORMATO_FECHA = '%Y-%m-%d'


def configurar_rutas(base):
    """Define la carpeta de datos y las rutas de cada archivo de persistencia."""
    global RUTA_BASE, RUTA_USUARIOS, RUTA_LIBROS, RUTA_CATEGORIAS
    global RUTA_PRESTAMOS, RUTA_MULTAS, RUTA_RESERVAS
    RUTA_BASE = os.path.join(base, '')
    os.makedirs(RUTA_BASE, exist_ok=True)
    RUTA_USUARIOS = RUTA_BASE + 'usuarios.txt'
    RUTA_LIBROS = RUTA_BASE + 'libros.txt'
    RUTA_CATEGORIAS = RUTA_BASE + 'categorias.txt'
    RUTA_PRESTAMOS = RUTA_BASE + 'prestamos.txt'
    RUTA_MULTAS = RUTA_BASE + 'multas.txt'
    RUTA_RESERVAS = RUTA_BASE + 'reservas.txt'


configurar_rutas(RUTA_BASE)
print('Carpeta de datos:', RUTA_BASE)

---
# SECCIÓN 1 — MODELO (dominio)
Cada entidad se representa con un **diccionario**. Por cada entidad hay tres funciones:
`crear_<entidad>()` (valida y construye), `<entidad>_to_linea()` (dict → texto) y
`<entidad>_from_linea()` (texto → dict).

## Celda 1.1 — Enums y validaciones comunes

In [ ]:
# ---- "Enums" del dominio (tuplas de valores permitidos) ----
TIPO_USUARIO_ENUM = ('ESTUDIANTE', 'DOCENTE', 'EXTERNO')
ESTADO_USUARIO_ENUM = ('ACTIVO', 'SUSPENDIDO')
ESTADO_PRESTAMO_ENUM = ('ACTIVO', 'DEVUELTO', 'VENCIDO')
ESTADO_MULTA_ENUM = ('PENDIENTE', 'PAGADA')
ESTADO_RESERVA_ENUM = ('ACTIVA', 'CUMPLIDA', 'CANCELADA')

# Reglas por tipo de usuario: días de préstamo y cupo máximo de libros
REGLAS_POR_TIPO = {
    'ESTUDIANTE': {'dias': 7, 'cupo': 3},
    'DOCENTE': {'dias': 15, 'cupo': 5},
    'EXTERNO': {'dias': 5, 'cupo': 2},
}


def validar_texto(valor, campo, obligatorio=True, max_len=100):
    """Limpia un texto y verifica que no esté vacío, no sea muy largo ni contenga el separador '|'."""
    texto = ' '.join(('' if valor is None else str(valor)).split())
    if obligatorio and not texto:
        raise ValueError(f'El campo "{campo}" es obligatorio')
    if SEPARADOR in texto:
        raise ValueError(f'El campo "{campo}" no puede contener el carácter "{SEPARADOR}"')
    if len(texto) > max_len:
        raise ValueError(f'El campo "{campo}" admite máximo {max_len} caracteres')
    return texto


def validar_solo_letras(valor, campo, max_len=80):
    """Texto que solo admite letras (con tildes y ñ), espacios y los signos . ' -"""
    texto = validar_texto(valor, campo, max_len=max_len)
    if not texto[0].isalpha() or not all(ch.isalpha() or ch in " .'-" for ch in texto):
        raise ValueError(f'El campo "{campo}" solo admite letras (sin números ni símbolos)')
    return texto


def validar_solo_digitos(valor, campo, min_len=1, max_len=20):
    """Texto que solo admite dígitos 0-9, con una longitud entre min_len y max_len."""
    texto = validar_texto(valor, campo).replace(' ', '')
    if not (texto.isascii() and texto.isdigit()):
        raise ValueError(f'El campo "{campo}" solo admite números (sin letras, espacios ni símbolos)')
    if not min_len <= len(texto) <= max_len:
        rango = f'{min_len}' if min_len == max_len else f'entre {min_len} y {max_len}'
        raise ValueError(f'El campo "{campo}" debe tener {rango} dígitos (tiene {len(texto)})')
    return texto


def validar_correo(valor):
    """Correo con formato usuario@dominio.ext"""
    texto = validar_texto(valor, 'correo').lower()
    if not re.fullmatch(r'[\w.+-]+@[\w-]+(\.[\w-]+)+', texto):
        raise ValueError(f'Correo inválido: "{texto}" (ejemplo válido: nombre@correo.com)')
    return texto


def validar_opcion(valor, opciones, campo):
    """Verifica que el valor (sin importar mayúsculas) pertenezca a un enum."""
    texto = validar_texto(valor, campo).upper()
    if texto not in opciones:
        raise ValueError(f'{campo} inválido: "{valor}". Opciones: {", ".join(opciones)}')
    return texto


def validar_entero(valor, campo, minimo=None, maximo=None):
    """Convierte un valor a entero y verifica que esté en el rango indicado."""
    try:
        numero = int(str(valor).strip())
    except ValueError:
        raise ValueError(f'El campo "{campo}" debe ser un número entero') from None
    if minimo is not None and numero < minimo:
        raise ValueError(f'El campo "{campo}" debe ser mayor o igual a {minimo}')
    if maximo is not None and numero > maximo:
        raise ValueError(f'El campo "{campo}" debe ser menor o igual a {maximo}')
    return numero


def validar_fecha(valor, campo):
    """Acepta un date o un texto 'AAAA-MM-DD' y retorna la fecha como texto ISO."""
    if isinstance(valor, date):
        return valor.strftime(FORMATO_FECHA)
    try:
        return datetime.strptime(str(valor).strip(), FORMATO_FECHA).strftime(FORMATO_FECHA)
    except ValueError:
        raise ValueError(f'El campo "{campo}" debe tener formato AAAA-MM-DD') from None


def a_fecha(texto):
    """Convierte un texto 'AAAA-MM-DD' a date."""
    return datetime.strptime(texto, FORMATO_FECHA).date()


def a_linea(valores):
    """Une valores con el separador; los vacíos se guardan como NINGUNO."""
    return SEPARADOR.join(VACIO if v is None or v == '' else str(v) for v in valores)


def partir_linea(linea, cantidad, entidad):
    """Divide una línea del archivo y convierte NINGUNO en None."""
    campos = linea.split(SEPARADOR)
    if len(campos) != cantidad:
        raise ValueError(f'Línea de {entidad} con formato inválido: {linea}')
    return [None if c == VACIO else c for c in campos]

## Celda 1.2 — Usuario
`NUM_DOC|NOMBRE|TIPO|CORREO|TELEFONO|ESTADO`

In [ ]:
def crear_usuario(num_documento, nombre, tipo, correo, telefono, estado='ACTIVO'):
    """Construye y valida el diccionario de un usuario."""
    return {
        'num_documento': validar_solo_digitos(num_documento, 'número de documento', 5, 12),
        'nombre': validar_solo_letras(nombre, 'nombre'),
        'tipo': validar_opcion(tipo, TIPO_USUARIO_ENUM, 'Tipo de usuario'),
        'correo': validar_correo(correo),
        'telefono': validar_solo_digitos(telefono, 'teléfono', 7, 10),
        'estado': validar_opcion(estado, ESTADO_USUARIO_ENUM, 'Estado de usuario'),
    }


def usuario_to_linea(u):
    """Convierte un usuario en una línea de usuarios.txt."""
    return a_linea([u['num_documento'], u['nombre'], u['tipo'], u['correo'], u['telefono'], u['estado']])


def usuario_from_linea(linea):
    """Convierte una línea de usuarios.txt en un usuario."""
    return crear_usuario(*partir_linea(linea, 6, 'usuario'))

## Celda 1.3 — Categoria
`CODIGO|NOMBRE|DESCRIPCION`

In [ ]:
def crear_categoria(codigo, nombre, descripcion=None):
    """Construye y valida el diccionario de una categoría."""
    codigo = validar_texto(codigo, 'código de categoría').upper()
    if not (codigo.isascii() and codigo.isalpha() and 2 <= len(codigo) <= 5):
        raise ValueError('El código de categoría debe tener de 2 a 5 letras, sin números ni espacios (ej. LIT)')
    return {
        'codigo': codigo,
        'nombre': validar_solo_letras(nombre, 'nombre de categoría', max_len=40),
        'descripcion': validar_texto(descripcion, 'descripción', obligatorio=False) or None,
    }


def categoria_to_linea(c):
    """Convierte una categoría en una línea de categorias.txt."""
    return a_linea([c['codigo'], c['nombre'], c['descripcion']])


def categoria_from_linea(linea):
    """Convierte una línea de categorias.txt en una categoría."""
    return crear_categoria(*partir_linea(linea, 3, 'categoría'))

## Celda 1.4 — Libro
`ISBN|TITULO|AUTOR|COD_CATEGORIA|ANIO|TOTAL_EJEMPLARES|DISPONIBLES`

In [ ]:
def crear_libro(isbn, titulo, autor, cod_categoria, anio, total_ejemplares, disponibles=None):
    """Construye y valida el diccionario de un libro (disponibles = total si no se indica)."""
    isbn = validar_solo_digitos(str(isbn).replace('-', ''), 'ISBN', 10, 13)
    if len(isbn) not in (10, 13):
        raise ValueError(f'El ISBN debe tener 10 o 13 dígitos (tiene {len(isbn)})')
    total = validar_entero(total_ejemplares, 'total de ejemplares', minimo=1, maximo=999)
    disponibles = total if disponibles is None else validar_entero(
        disponibles, 'ejemplares disponibles', minimo=0, maximo=total)
    return {
        'isbn': isbn,
        'titulo': validar_texto(titulo, 'título'),
        'autor': validar_solo_letras(autor, 'autor'),
        'cod_categoria': validar_texto(cod_categoria, 'categoría').upper(),
        'anio': validar_entero(anio, 'año', minimo=1000, maximo=date.today().year),
        'total_ejemplares': total,
        'disponibles': disponibles,
    }


def libro_to_linea(l):
    """Convierte un libro en una línea de libros.txt."""
    return a_linea([l['isbn'], l['titulo'], l['autor'], l['cod_categoria'],
                    l['anio'], l['total_ejemplares'], l['disponibles']])


def libro_from_linea(linea):
    """Convierte una línea de libros.txt en un libro."""
    return crear_libro(*partir_linea(linea, 7, 'libro'))

## Celda 1.5 — Prestamo
`CODIGO|ISBN|NUM_DOC|FECHA_PRESTAMO|FECHA_LIMITE|FECHA_DEVOLUCION|ESTADO`

In [ ]:
def crear_prestamo(codigo, isbn, num_documento, fecha_prestamo, fecha_limite,
                   fecha_devolucion=None, estado='ACTIVO'):
    """Construye y valida el diccionario de un préstamo."""
    return {
        'codigo': validar_texto(codigo, 'código de préstamo'),
        'isbn': validar_texto(isbn, 'ISBN'),
        'num_documento': validar_texto(num_documento, 'número de documento'),
        'fecha_prestamo': validar_fecha(fecha_prestamo, 'fecha de préstamo'),
        'fecha_limite': validar_fecha(fecha_limite, 'fecha límite'),
        'fecha_devolucion': validar_fecha(fecha_devolucion, 'fecha de devolución') if fecha_devolucion else None,
        'estado': validar_opcion(estado, ESTADO_PRESTAMO_ENUM, 'Estado de préstamo'),
    }


def prestamo_to_linea(p):
    """Convierte un préstamo en una línea de prestamos.txt."""
    return a_linea([p['codigo'], p['isbn'], p['num_documento'], p['fecha_prestamo'],
                    p['fecha_limite'], p['fecha_devolucion'], p['estado']])


def prestamo_from_linea(linea):
    """Convierte una línea de prestamos.txt en un préstamo."""
    return crear_prestamo(*partir_linea(linea, 7, 'préstamo'))

## Celda 1.6 — Multa
`CODIGO|COD_PRESTAMO|DIAS_RETRASO|VALOR|ESTADO`

In [ ]:
def crear_multa(codigo, cod_prestamo, dias_retraso, valor, estado='PENDIENTE'):
    """Construye y valida el diccionario de una multa."""
    return {
        'codigo': validar_texto(codigo, 'código de multa'),
        'cod_prestamo': validar_texto(cod_prestamo, 'código de préstamo'),
        'dias_retraso': validar_entero(dias_retraso, 'días de retraso', minimo=1),
        'valor': validar_entero(valor, 'valor', minimo=0),
        'estado': validar_opcion(estado, ESTADO_MULTA_ENUM, 'Estado de multa'),
    }


def multa_to_linea(m):
    """Convierte una multa en una línea de multas.txt."""
    return a_linea([m['codigo'], m['cod_prestamo'], m['dias_retraso'], m['valor'], m['estado']])


def multa_from_linea(linea):
    """Convierte una línea de multas.txt en una multa."""
    return crear_multa(*partir_linea(linea, 5, 'multa'))

## Celda 1.7 — Reserva
`CODIGO|ISBN|NUM_DOC|FECHA_RESERVA|ESTADO`

In [ ]:
def crear_reserva(codigo, isbn, num_documento, fecha_reserva, estado='ACTIVA'):
    """Construye y valida el diccionario de una reserva."""
    return {
        'codigo': validar_texto(codigo, 'código de reserva'),
        'isbn': validar_texto(isbn, 'ISBN'),
        'num_documento': validar_texto(num_documento, 'número de documento'),
        'fecha_reserva': validar_fecha(fecha_reserva, 'fecha de reserva'),
        'estado': validar_opcion(estado, ESTADO_RESERVA_ENUM, 'Estado de reserva'),
    }


def reserva_to_linea(r):
    """Convierte una reserva en una línea de reservas.txt."""
    return a_linea([r['codigo'], r['isbn'], r['num_documento'], r['fecha_reserva'], r['estado']])


def reserva_from_linea(linea):
    """Convierte una línea de reservas.txt en una reserva."""
    return crear_reserva(*partir_linea(linea, 5, 'reserva'))

---
# SECCIÓN 2 — REPOSITORIOS (persistencia)
Leen y escriben los archivos `.txt` de Google Drive. Ninguna función de esta capa aplica reglas de negocio.

## Celda 2.1 — ArchivoUtil

In [ ]:
_ultimo_codigo = ''


def leer_lineas(ruta):
    """Lee todas las líneas no vacías de un archivo. Retorna lista vacía si no existe."""
    try:
        with open(ruta, 'r', encoding='utf-8') as f:
            return [l.strip() for l in f if l.strip()]
    except FileNotFoundError:
        return []


def escribir_linea(ruta, linea):
    """Agrega una línea al final del archivo."""
    with open(ruta, 'a', encoding='utf-8') as f:
        f.write(linea + '\n')


def reescribir(ruta, lineas):
    """Reemplaza el contenido completo del archivo."""
    with open(ruta, 'w', encoding='utf-8') as f:
        f.write(''.join(l + '\n' for l in lineas))


def generar_codigo(prefijo):
    """Genera un código único basado en timestamp, p. ej. PRE-20251015083000123."""
    global _ultimo_codigo
    ts = datetime.now().strftime('%Y%m%d%H%M%S%f')[:17]
    if ts <= _ultimo_codigo:            # dos códigos en el mismo milisegundo
        ts = str(int(_ultimo_codigo) + 1)
    _ultimo_codigo = ts
    return f'{prefijo}-{ts}'


def leer_registros(ruta, from_linea):
    """Lee un archivo y convierte cada línea en diccionario."""
    return [from_linea(l) for l in leer_lineas(ruta)]


def reescribir_registros(ruta, registros, to_linea):
    """Sobrescribe un archivo con una lista de diccionarios."""
    reescribir(ruta, [to_linea(r) for r in registros])


def reemplazar_registro(ruta, from_linea, to_linea, clave, nuevo):
    """Reemplaza en el archivo el registro cuya clave coincide con la del nuevo."""
    registros = leer_registros(ruta, from_linea)
    for i, r in enumerate(registros):
        if r[clave] == nuevo[clave]:
            registros[i] = nuevo
            reescribir_registros(ruta, registros, to_linea)
            return
    raise ValueError(f'No existe el registro con {clave} = {nuevo[clave]}')

## Celda 2.2 — UsuarioRepository

In [ ]:
def repo_usuario_guardar(u):
    """Agrega un usuario a usuarios.txt."""
    escribir_linea(RUTA_USUARIOS, usuario_to_linea(u))


def repo_usuario_find_by_documento(doc):
    """Retorna el usuario con ese documento o None."""
    return next((u for u in repo_usuario_find_all() if u['num_documento'] == doc), None)


def repo_usuario_find_all():
    """Retorna todos los usuarios."""
    return leer_registros(RUTA_USUARIOS, usuario_from_linea)


def repo_usuario_actualizar(u):
    """Reescribe los datos de un usuario existente."""
    reemplazar_registro(RUTA_USUARIOS, usuario_from_linea, usuario_to_linea, 'num_documento', u)

## Celda 2.3 — CategoriaRepository

In [ ]:
def repo_categoria_guardar(c):
    """Agrega una categoría a categorias.txt."""
    escribir_linea(RUTA_CATEGORIAS, categoria_to_linea(c))


def repo_categoria_find_by_codigo(cod):
    """Retorna la categoría con ese código o None."""
    cod = str(cod).strip().upper()
    return next((c for c in repo_categoria_find_all() if c['codigo'] == cod), None)


def repo_categoria_find_all():
    """Retorna todas las categorías."""
    return leer_registros(RUTA_CATEGORIAS, categoria_from_linea)

## Celda 2.4 — LibroRepository

In [ ]:
def repo_libro_guardar(l):
    """Agrega un libro a libros.txt."""
    escribir_linea(RUTA_LIBROS, libro_to_linea(l))


def repo_libro_find_by_isbn(isbn):
    """Retorna el libro con ese ISBN o None."""
    isbn = str(isbn).strip().replace('-', '')
    return next((l for l in repo_libro_find_all() if l['isbn'] == isbn), None)


def repo_libro_find_by_titulo(txt):
    """Retorna los libros cuyo título contiene el texto (sin distinguir mayúsculas)."""
    txt = str(txt).strip().lower()
    return [l for l in repo_libro_find_all() if txt in l['titulo'].lower()]


def repo_libro_find_by_categoria(cod):
    """Retorna los libros de una categoría."""
    cod = str(cod).strip().upper()
    return [l for l in repo_libro_find_all() if l['cod_categoria'] == cod]


def repo_libro_find_all():
    """Retorna todos los libros."""
    return leer_registros(RUTA_LIBROS, libro_from_linea)


def repo_libro_actualizar(l):
    """Reescribe los datos de un libro existente."""
    reemplazar_registro(RUTA_LIBROS, libro_from_linea, libro_to_linea, 'isbn', l)

## Celda 2.5 — PrestamoRepository

In [ ]:
def repo_prestamo_guardar(p):
    """Agrega un préstamo a prestamos.txt."""
    escribir_linea(RUTA_PRESTAMOS, prestamo_to_linea(p))


def repo_prestamo_find_by_codigo(cod):
    """Retorna el préstamo con ese código o None."""
    cod = str(cod).strip().upper()
    return next((p for p in repo_prestamo_find_all() if p['codigo'] == cod), None)


def repo_prestamo_find_by_usuario(doc):
    """Retorna los préstamos de un usuario."""
    return [p for p in repo_prestamo_find_all() if p['num_documento'] == doc]


def repo_prestamo_find_by_libro(isbn):
    """Retorna los préstamos de un libro."""
    return [p for p in repo_prestamo_find_all() if p['isbn'] == isbn]


def repo_prestamo_find_all():
    """Retorna todos los préstamos."""
    return leer_registros(RUTA_PRESTAMOS, prestamo_from_linea)


def repo_prestamo_actualizar(p):
    """Reescribe los datos de un préstamo existente."""
    reemplazar_registro(RUTA_PRESTAMOS, prestamo_from_linea, prestamo_to_linea, 'codigo', p)


def repo_prestamo_reescribir_todos(prestamos):
    """Sobrescribe prestamos.txt con la lista recibida (actualización masiva)."""
    reescribir_registros(RUTA_PRESTAMOS, prestamos, prestamo_to_linea)

## Celda 2.6 — MultaRepository

In [ ]:
def repo_multa_guardar(m):
    """Agrega una multa a multas.txt."""
    escribir_linea(RUTA_MULTAS, multa_to_linea(m))


def repo_multa_find_by_codigo(cod):
    """Retorna la multa con ese código o None."""
    cod = str(cod).strip().upper()
    return next((m for m in repo_multa_find_all() if m['codigo'] == cod), None)


def repo_multa_find_by_prestamo(cod):
    """Retorna la multa generada por un préstamo o None."""
    return next((m for m in repo_multa_find_all() if m['cod_prestamo'] == cod), None)


def repo_multa_find_pendientes_usuario(doc):
    """Retorna las multas PENDIENTES de un usuario (cruza multas con préstamos)."""
    codigos = {p['codigo'] for p in repo_prestamo_find_by_usuario(doc)}
    return [m for m in repo_multa_find_all()
            if m['cod_prestamo'] in codigos and m['estado'] == 'PENDIENTE']


def repo_multa_find_all():
    """Retorna todas las multas."""
    return leer_registros(RUTA_MULTAS, multa_from_linea)


def repo_multa_actualizar_estado(cod, estado):
    """Cambia el estado de una multa."""
    m = repo_multa_find_by_codigo(cod)
    if m is None:
        raise ValueError(f'No existe la multa {cod}')
    m['estado'] = validar_opcion(estado, ESTADO_MULTA_ENUM, 'Estado de multa')
    reemplazar_registro(RUTA_MULTAS, multa_from_linea, multa_to_linea, 'codigo', m)

## Celda 2.7 — ReservaRepository

In [ ]:
def repo_reserva_guardar(r):
    """Agrega una reserva a reservas.txt."""
    escribir_linea(RUTA_RESERVAS, reserva_to_linea(r))


def repo_reserva_find_by_codigo(cod):
    """Retorna la reserva con ese código o None."""
    cod = str(cod).strip().upper()
    return next((r for r in repo_reserva_find_all() if r['codigo'] == cod), None)


def repo_reserva_find_by_libro(isbn):
    """Retorna las reservas de un libro."""
    return [r for r in repo_reserva_find_all() if r['isbn'] == isbn]


def repo_reserva_find_by_usuario(doc):
    """Retorna las reservas de un usuario."""
    return [r for r in repo_reserva_find_all() if r['num_documento'] == doc]


def repo_reserva_find_all():
    """Retorna todas las reservas."""
    return leer_registros(RUTA_RESERVAS, reserva_from_linea)


def repo_reserva_actualizar_estado(cod, estado):
    """Cambia el estado de una reserva."""
    r = repo_reserva_find_by_codigo(cod)
    if r is None:
        raise ValueError(f'No existe la reserva {cod}')
    r['estado'] = validar_opcion(estado, ESTADO_RESERVA_ENUM, 'Estado de reserva')
    reemplazar_registro(RUTA_RESERVAS, reserva_from_linea, reserva_to_linea, 'codigo', r)

---
# SECCIÓN 3 — SERVICIOS (lógica de negocio)
Aplican las reglas del sistema y usan los repositorios. Ante un error lanzan `ValueError`
con un mensaje claro que la interfaz muestra al usuario.

## Celda 3.1 — UsuarioService (RF-01)

In [ ]:
def usuario_registrar(num_documento, nombre, tipo, correo, telefono):
    """Registra un usuario nuevo en estado ACTIVO."""
    u = crear_usuario(num_documento, nombre, tipo, correo, telefono)
    if repo_usuario_find_by_documento(u['num_documento']):
        raise ValueError(f'Ya existe un usuario con documento {u["num_documento"]}')
    repo_usuario_guardar(u)
    return u


def usuario_buscar_por_documento(num_documento):
    """Retorna el usuario o lanza error si no existe."""
    u = repo_usuario_find_by_documento(str(num_documento).strip())
    if u is None:
        raise ValueError(f'No existe un usuario con documento {num_documento}')
    return u


def usuario_actualizar(num_documento, nombre=None, correo=None, telefono=None, estado=None):
    """Actualiza datos personales, de contacto y/o estado (los None no se modifican)."""
    u = usuario_buscar_por_documento(num_documento)
    nuevo = crear_usuario(u['num_documento'], nombre or u['nombre'], u['tipo'],
                          correo or u['correo'], telefono or u['telefono'], estado or u['estado'])
    repo_usuario_actualizar(nuevo)
    return nuevo


def usuario_listar_todos():
    """Retorna todos los usuarios ordenados por nombre."""
    return sorted(repo_usuario_find_all(), key=lambda u: u['nombre'].lower())

## Celda 3.2 — LibroService (RF-02)

In [ ]:
def libro_registrar_categoria(codigo, nombre, descripcion=None):
    """Registra una categoría nueva (código único, p. ej. LIT, TEC)."""
    c = crear_categoria(codigo, nombre, descripcion)
    if repo_categoria_find_by_codigo(c['codigo']):
        raise ValueError(f'Ya existe la categoría {c["codigo"]}')
    repo_categoria_guardar(c)
    return c


def libro_listar_categorias():
    """Retorna todas las categorías ordenadas por código."""
    return sorted(repo_categoria_find_all(), key=lambda c: c['codigo'])


def libro_registrar(isbn, titulo, autor, cod_categoria, anio, total_ejemplares):
    """Registra un libro nuevo; todos sus ejemplares quedan disponibles."""
    l = crear_libro(isbn, titulo, autor, cod_categoria, anio, total_ejemplares)
    if repo_libro_find_by_isbn(l['isbn']):
        raise ValueError(f'Ya existe un libro con ISBN {l["isbn"]}')
    if repo_categoria_find_by_codigo(l['cod_categoria']) is None:
        raise ValueError(f'No existe la categoría {l["cod_categoria"]}')
    repo_libro_guardar(l)
    return l


def libro_buscar_por_isbn(isbn):
    """Retorna el libro o lanza error si no existe."""
    l = repo_libro_find_by_isbn(isbn)
    if l is None:
        raise ValueError(f'No existe un libro con ISBN {isbn}')
    return l


def libro_buscar_por_titulo(texto):
    """Retorna los libros cuyo título contiene el texto."""
    return repo_libro_find_by_titulo(texto)


def libro_listar_por_categoria(cod_categoria):
    """Retorna los libros de una categoría existente."""
    if repo_categoria_find_by_codigo(cod_categoria) is None:
        raise ValueError(f'No existe la categoría {cod_categoria}')
    return repo_libro_find_by_categoria(cod_categoria)


def libro_listar_disponibles():
    """Retorna los libros con al menos un ejemplar disponible."""
    return [l for l in repo_libro_find_all() if l['disponibles'] > 0]

## Celda 3.3 — PrestamoService (RF-03 y historial RF-04)
Reglas: usuario **ACTIVO**, **sin multas pendientes**, dentro de su **cupo**, con ejemplar disponible y
respetando la **cola de reservas** (por orden de llegada).

In [ ]:
def prestamo_actualizar_vencidos(fecha_ref=None):
    """Marca como VENCIDO todo préstamo ACTIVO cuya fecha límite ya pasó."""
    ref = validar_fecha(fecha_ref or date.today(), 'fecha de referencia')
    prestamos = repo_prestamo_find_all()
    cambios = 0
    for p in prestamos:
        if p['estado'] == 'ACTIVO' and p['fecha_limite'] < ref:
            p['estado'] = 'VENCIDO'
            cambios += 1
    if cambios:
        repo_prestamo_reescribir_todos(prestamos)
    return cambios


def prestamo_prestar(num_documento, isbn, fecha=None):
    """Registra un préstamo validando todas las reglas. Retorna el préstamo creado."""
    fecha = a_fecha(validar_fecha(fecha or date.today(), 'fecha de préstamo'))
    if fecha > date.today():
        raise ValueError('La fecha de préstamo no puede ser futura')
    usuario = usuario_buscar_por_documento(num_documento)
    doc = usuario['num_documento']
    if usuario['estado'] != 'ACTIVO':
        raise ValueError(f'El usuario {usuario["nombre"]} está SUSPENDIDO')
    pendientes = multa_listar_pendientes_usuario(doc)
    if pendientes:
        total = sum(m['valor'] for m in pendientes)
        raise ValueError(f'El usuario tiene {len(pendientes)} multa(s) pendiente(s) por ${total:,} COP')

    libro = libro_buscar_por_isbn(isbn)
    prestamo_actualizar_vencidos()
    reglas = REGLAS_POR_TIPO[usuario['tipo']]
    abiertos = [p for p in repo_prestamo_find_by_usuario(doc) if p['estado'] != 'DEVUELTO']
    if len(abiertos) >= reglas['cupo']:
        raise ValueError(f'El usuario alcanzó su cupo de {reglas["cupo"]} libros ({usuario["tipo"]})')
    if any(p['isbn'] == libro['isbn'] for p in abiertos):
        raise ValueError('El usuario ya tiene prestado un ejemplar de este libro')
    if libro['disponibles'] <= 0:
        raise ValueError('No hay ejemplares disponibles. Puede reservar el libro')

    # Cola de reservas: los ejemplares disponibles se entregan por orden de llegada
    cola = reserva_listar_por_libro(libro['isbn'])
    en_cola = [r['num_documento'] for r in cola]
    if doc in en_cola:
        posicion = en_cola.index(doc)
        if posicion >= libro['disponibles']:
            raise ValueError(f'Hay {posicion} reserva(s) anteriores a la suya para este libro')
        repo_reserva_actualizar_estado(cola[posicion]['codigo'], 'CUMPLIDA')
    elif len(cola) >= libro['disponibles']:
        raise ValueError('Los ejemplares disponibles están apartados para usuarios con reserva')

    p = crear_prestamo(generar_codigo('PRE'), libro['isbn'], doc, fecha,
                       fecha + timedelta(days=reglas['dias']))
    repo_prestamo_guardar(p)
    libro['disponibles'] -= 1
    repo_libro_actualizar(libro)
    return p


def prestamo_devolver(codigo, fecha=None):
    """Registra la devolución, libera el ejemplar y genera multa si hay retraso.
    Retorna (prestamo, multa) donde multa es None si se devolvió a tiempo."""
    p = prestamo_buscar_por_codigo(codigo)
    if p['estado'] == 'DEVUELTO':
        raise ValueError('Ese préstamo ya fue devuelto')
    fecha_dev = validar_fecha(fecha or date.today(), 'fecha de devolución')
    if fecha_dev < p['fecha_prestamo']:
        raise ValueError(f'La fecha de devolución no puede ser anterior a la de préstamo ({p["fecha_prestamo"]})')
    if fecha_dev > validar_fecha(date.today(), 'hoy'):
        raise ValueError('La fecha de devolución no puede ser futura')
    p['fecha_devolucion'] = fecha_dev
    p['estado'] = 'DEVUELTO'
    repo_prestamo_actualizar(p)

    libro = libro_buscar_por_isbn(p['isbn'])
    libro['disponibles'] = min(libro['disponibles'] + 1, libro['total_ejemplares'])
    repo_libro_actualizar(libro)
    return p, multa_calcular(p)


def prestamo_buscar_por_codigo(codigo):
    """Retorna el préstamo o lanza error si no existe."""
    p = repo_prestamo_find_by_codigo(codigo)
    if p is None:
        raise ValueError(f'No existe el préstamo {codigo}')
    return p


def prestamo_listar_activos():
    """Retorna los préstamos no devueltos (ACTIVO o VENCIDO)."""
    prestamo_actualizar_vencidos()
    return [p for p in repo_prestamo_find_all() if p['estado'] != 'DEVUELTO']


def prestamo_listar_por_usuario(num_documento):
    """Retorna los préstamos de un usuario ordenados por fecha."""
    usuario_buscar_por_documento(num_documento)
    prestamo_actualizar_vencidos()
    return sorted(repo_prestamo_find_by_usuario(str(num_documento).strip()),
                  key=lambda p: (p['fecha_prestamo'], p['codigo']))


def prestamo_historial_usuario(num_documento):
    """Historial completo de préstamos de un usuario en orden cronológico (RF-04)."""
    return prestamo_listar_por_usuario(num_documento)


def prestamo_listar_vencidos(fecha_ref=None):
    """Préstamos sin devolver cuya fecha límite es anterior a la fecha dada (hoy por defecto)."""
    ref = validar_fecha(fecha_ref or date.today(), 'fecha de referencia')
    prestamo_actualizar_vencidos()
    return sorted([p for p in repo_prestamo_find_all()
                   if p['estado'] != 'DEVUELTO' and p['fecha_limite'] < ref],
                  key=lambda p: p['fecha_limite'])

## Celda 3.4 — MultaService (RF-04)
COP 1.000 por cada día de retraso, calculada automáticamente al devolver.

In [ ]:
def multa_calcular(prestamo):
    """Genera y guarda la multa de un préstamo devuelto con retraso. Retorna la multa o None."""
    if not prestamo['fecha_devolucion']:
        raise ValueError('El préstamo aún no ha sido devuelto')
    dias = (a_fecha(prestamo['fecha_devolucion']) - a_fecha(prestamo['fecha_limite'])).days
    if dias <= 0:
        return None
    existente = repo_multa_find_by_prestamo(prestamo['codigo'])
    if existente:
        return existente
    m = crear_multa(generar_codigo('MUL'), prestamo['codigo'], dias, dias * MULTA_POR_DIA)
    repo_multa_guardar(m)
    return m


def multa_pagar(codigo):
    """Registra el pago de una multa pendiente."""
    m = repo_multa_find_by_codigo(codigo)
    if m is None:
        raise ValueError(f'No existe la multa {codigo}')
    if m['estado'] == 'PAGADA':
        raise ValueError('La multa ya estaba pagada')
    repo_multa_actualizar_estado(m['codigo'], 'PAGADA')
    m['estado'] = 'PAGADA'
    return m


def multa_listar_pendientes_usuario(num_documento):
    """Retorna las multas pendientes de un usuario."""
    return repo_multa_find_pendientes_usuario(str(num_documento).strip())


def multa_listar_por_usuario(num_documento):
    """Retorna todas las multas (pagadas y pendientes) de un usuario."""
    codigos = {p['codigo'] for p in repo_prestamo_find_by_usuario(str(num_documento).strip())}
    return [m for m in repo_multa_find_all() if m['cod_prestamo'] in codigos]

## Celda 3.5 — ReservaService (RF-04)
Solo se reserva un libro **sin ejemplares libres**; las reservas se atienden por orden de llegada.

In [ ]:
def reserva_reservar(num_documento, isbn, fecha=None):
    """Crea una reserva ACTIVA si el libro no tiene ejemplares libres. Retorna la reserva."""
    usuario = usuario_buscar_por_documento(num_documento)
    doc = usuario['num_documento']
    if usuario['estado'] != 'ACTIVO':
        raise ValueError('Un usuario SUSPENDIDO no puede reservar')
    libro = libro_buscar_por_isbn(isbn)
    cola = reserva_listar_por_libro(libro['isbn'])
    if libro['disponibles'] > len(cola):
        raise ValueError('El libro tiene ejemplares disponibles: puede prestarlo directamente')
    if any(r['num_documento'] == doc for r in cola):
        raise ValueError('El usuario ya tiene una reserva activa de este libro')
    if any(p['isbn'] == libro['isbn'] and p['estado'] != 'DEVUELTO'
           for p in repo_prestamo_find_by_usuario(doc)):
        raise ValueError('El usuario ya tiene prestado este libro')
    r = crear_reserva(generar_codigo('RES'), libro['isbn'], doc, fecha or date.today())
    repo_reserva_guardar(r)
    return r


def reserva_cancelar(codigo):
    """Cancela una reserva ACTIVA."""
    r = repo_reserva_find_by_codigo(codigo)
    if r is None or r['estado'] != 'ACTIVA':
        raise ValueError(f'No existe una reserva ACTIVA con código {codigo}')
    repo_reserva_actualizar_estado(r['codigo'], 'CANCELADA')
    r['estado'] = 'CANCELADA'
    return r


def reserva_listar_por_libro(isbn):
    """Cola de reservas ACTIVAS de un libro, por orden de llegada."""
    return sorted([r for r in repo_reserva_find_by_libro(str(isbn).strip()) if r['estado'] == 'ACTIVA'],
                  key=lambda r: (r['fecha_reserva'], r['codigo']))


def reserva_listar_por_usuario(num_documento):
    """Todas las reservas de un usuario ordenadas por fecha."""
    return sorted(repo_reserva_find_by_usuario(str(num_documento).strip()),
                  key=lambda r: (r['fecha_reserva'], r['codigo']))

## Celda 3.6 — AnalisisDatosService (RF-05)
Indicadores estadísticos con `collections.Counter` y `datetime` (solo librería estándar).

In [ ]:
def analisis_libros_mas_prestados(fecha_ini, fecha_fin, top_n=10):
    """Top N libros más prestados en el periodo. Lista de tuplas (isbn, titulo, veces)."""
    ini = validar_fecha(fecha_ini, 'fecha inicial')
    fin = validar_fecha(fecha_fin, 'fecha final')
    if ini > fin:
        raise ValueError('La fecha inicial no puede ser posterior a la final')
    titulos = {l['isbn']: l['titulo'] for l in repo_libro_find_all()}
    contador = Counter(p['isbn'] for p in repo_prestamo_find_all()
                       if ini <= p['fecha_prestamo'] <= fin)
    return [(isbn, titulos.get(isbn, 'Desconocido'), n) for isbn, n in contador.most_common(top_n)]


def analisis_prestamos_por_categoria():
    """Distribución de préstamos por categoría. Lista de (codigo, nombre, cantidad, porcentaje)."""
    cat_de_libro = {l['isbn']: l['cod_categoria'] for l in repo_libro_find_all()}
    nombres = {c['codigo']: c['nombre'] for c in repo_categoria_find_all()}
    contador = Counter(cat_de_libro.get(p['isbn'], 'SIN') for p in repo_prestamo_find_all())
    total = sum(contador.values())
    return [(cod, nombres.get(cod, 'Sin categoría'), n, round(n * 100 / total, 1))
            for cod, n in contador.most_common()]


def analisis_morosidad_por_tipo():
    """Tasa de morosidad por tipo de usuario. Dict tipo -> (total, morosos, tasa %).
    Moroso = préstamo VENCIDO o devuelto después de la fecha límite."""
    prestamo_actualizar_vencidos()
    tipo_de = {u['num_documento']: u['tipo'] for u in repo_usuario_find_all()}
    resultado = {}
    for tipo in TIPO_USUARIO_ENUM:
        prestamos = [p for p in repo_prestamo_find_all() if tipo_de.get(p['num_documento']) == tipo]
        morosos = [p for p in prestamos if p['estado'] == 'VENCIDO' or
                   (p['fecha_devolucion'] and p['fecha_devolucion'] > p['fecha_limite'])]
        tasa = round(len(morosos) * 100 / len(prestamos), 1) if prestamos else 0.0
        resultado[tipo] = (len(prestamos), len(morosos), tasa)
    return resultado


def analisis_duracion_promedio_prestamo():
    """Promedio de días de retención de los préstamos DEVUELTOS (None si no hay)."""
    dias = [(a_fecha(p['fecha_devolucion']) - a_fecha(p['fecha_prestamo'])).days
            for p in repo_prestamo_find_all() if p['estado'] == 'DEVUELTO']
    return round(sum(dias) / len(dias), 1) if dias else None


def analisis_reporte_multas():
    """Valor total generado, pagado y pendiente de las multas, con sus cantidades."""
    multas = repo_multa_find_all()
    pagadas = [m for m in multas if m['estado'] == 'PAGADA']
    pendientes = [m for m in multas if m['estado'] == 'PENDIENTE']
    return {
        'cantidad': len(multas), 'generado': sum(m['valor'] for m in multas),
        'cant_pagadas': len(pagadas), 'pagado': sum(m['valor'] for m in pagadas),
        'cant_pendientes': len(pendientes), 'pendiente': sum(m['valor'] for m in pendientes),
    }

---
# SECCIÓN 4 — INTERFAZ DE USUARIO
Menús con `print()` e `input()`. Los datos se piden con **formularios**:

* Cada campo se **valida en el momento**. Si el dato es inválido se explica el error y se vuelve a pedir
  **solo ese campo** (números, solo letras, correo, fechas, longitud, existencia en el sistema…).
* Los campos con opciones fijas (tipo de usuario, estado, categoría) muestran una **lista numerada**:
  se escribe el número o el nombre y solo se acepta una opción de la lista.
* Escriba **`<`** para **volver al campo anterior** y **`*`** para **cancelar** la operación.
* Al volver a un campo, el valor ya escrito aparece entre corchetes: **Enter** lo conserva.
* Antes de guardar se muestra un **resumen**: `s` guarda, un **número** corrige ese campo, `<` corrige el último.

## Celda 4.0 — Funciones auxiliares de consola y formularios

In [ ]:
ATRAS = '<'       # volver al campo anterior
CANCELAR = '*'    # cancelar el formulario


def limpiar_pantalla():
    """Limpia la salida de la celda (Colab/Jupyter) o la consola."""
    try:
        from IPython.display import clear_output
        clear_output(wait=True)
    except ImportError:
        os.system('cls' if os.name == 'nt' else 'clear')


def leer_texto(mensaje, obligatorio=True):
    """Pide un texto; si es obligatorio repite hasta que no esté vacío."""
    while True:
        texto = input(mensaje).strip()
        if texto or not obligatorio:
            return texto
        print('  Este dato es obligatorio.')


# ---------- Formularios: campos con validación inmediata, volver y cancelar ----------

def campo(clave, etiqueta, validar=None, opciones=None, por_defecto=None, opcional=False):
    """Describe un campo de formulario.
    validar(texto, valores) -> valor convertido, o lanza ValueError con el motivo.
    opciones: lista de (valor, texto) o función(valores) que la retorna; solo se acepta una de ellas.
    por_defecto: valor (o función(valores)) que se usa al presionar Enter."""
    return {'clave': clave, 'etiqueta': etiqueta, 'validar': validar, 'opciones': opciones,
            'por_defecto': por_defecto, 'opcional': opcional}


def resolver(valor, valores):
    """Si el valor es una función la evalúa con los valores ya ingresados."""
    return valor(valores) if callable(valor) else valor


def elegir_opcion(texto, opciones):
    """Convierte el número de la lista o el nombre escrito en una de las opciones válidas."""
    valores_validos = [v for v, _ in opciones]
    if texto.isdigit() and 1 <= int(texto) <= len(opciones):
        return valores_validos[int(texto) - 1]
    for v in valores_validos:
        if texto.upper() == str(v).upper():
            return v
    raise ValueError(f'"{texto}" no es una opción válida. Escriba un número del 1 al {len(opciones)}')


def leer_campo(c, valores, numero, total, previo=None):
    """Pide un campo hasta que sea válido. Retorna el valor, ATRAS o CANCELAR."""
    opciones = resolver(c['opciones'], valores)
    defecto = previo if previo is not None else resolver(c['por_defecto'], valores)
    if opciones:
        for i, (_, texto) in enumerate(opciones, 1):
            print(f'        {i}) {texto}')
    elif opciones is not None:
        print('        (no hay opciones registradas; escriba * para cancelar)')
    sufijo = f' [{defecto}]' if defecto not in (None, '') else (' (opcional, Enter = vacío)' if c['opcional'] else '')
    while True:
        texto = input(f'  [{numero}/{total}] {c["etiqueta"]}{sufijo}: ').strip()
        if texto in (ATRAS, CANCELAR):
            return texto
        if not texto:
            if defecto not in (None, ''):
                texto = str(defecto)
            elif c['opcional']:
                return ''
            else:
                print('      ✗ Este campo es obligatorio.')
                continue
        try:
            if opciones == []:
                raise ValueError('No hay opciones disponibles para este campo')
            valor = elegir_opcion(texto, opciones) if opciones else texto
            return c['validar'](valor, valores) if c['validar'] else valor
        except ValueError as e:
            print(f'      ✗ {e}. Corrija el dato ({ATRAS} = campo anterior, {CANCELAR} = cancelar)')


def leer_formulario(titulo, campos, confirmar=True):
    """Pide los campos en orden con validación inmediata.
    '<' vuelve al campo anterior y '*' cancela. Si confirmar=True muestra un resumen antes de
    guardar, donde se puede corregir cualquier campo. Retorna dict clave -> valor, o None si se cancela."""
    print(f'\n  --- {titulo} ---   ({ATRAS} = campo anterior · {CANCELAR} = cancelar)')
    valores, i = {}, 0
    while True:
        while i < len(campos):
            c = campos[i]
            r = leer_campo(c, valores, i + 1, len(campos), valores.get(c['clave']))
            if r == CANCELAR:
                print('  Operación cancelada.')
                return None
            if r == ATRAS:
                if i == 0:
                    print('      (ya está en el primer campo)')
                else:
                    i -= 1
                continue
            valores[c['clave']] = r
            i += 1
        if not confirmar:
            return valores
        print('  Resumen:')
        for n, c in enumerate(campos, 1):
            print(f'    {n}. {c["etiqueta"]}: {valores[c["clave"]] if valores[c["clave"]] != "" else "(vacío)"}')
        while True:
            r = input(f'  ¿Guardar? (s = sí · 1-{len(campos)} = corregir ese campo · '
                      f'{ATRAS} = corregir el último · {CANCELAR} = cancelar): ').strip().lower()
            if r in ('s', 'si', 'sí'):
                return valores
            if r == CANCELAR:
                print('  Operación cancelada.')
                return None
            if r == ATRAS:
                i = len(campos) - 1
                break
            if r.isdigit() and 1 <= int(r) <= len(campos):
                i = int(r) - 1
                break
            print('      ✗ Respuesta no válida.')


def leer_entero(mensaje, minimo=None, maximo=None):
    """Pide un número entero en un rango (None si se cancela)."""
    d = leer_formulario(mensaje, [campo('n', mensaje, v_entero(mensaje, minimo, maximo))], confirmar=False)
    return None if d is None else d['n']


def leer_fecha(mensaje, por_defecto=None):
    """Pide una fecha AAAA-MM-DD; Enter = fecha por defecto (hoy). None si se cancela."""
    d = leer_formulario(mensaje, [campo('f', mensaje, v_fecha(), por_defecto=por_defecto or date.today())],
                        confirmar=False)
    return None if d is None else d['f']


# ---------- Validadores de campo (reutilizan las validaciones del modelo) ----------

def v_letras(nombre_campo, max_len=80):
    """Campo de solo letras."""
    return lambda t, v: validar_solo_letras(t, nombre_campo, max_len)


def v_texto(nombre_campo, max_len=100):
    """Campo de texto libre (sin '|')."""
    return lambda t, v: validar_texto(t, nombre_campo, max_len=max_len)


def v_digitos(nombre_campo, min_len, max_len):
    """Campo de solo números con longitud."""
    return lambda t, v: validar_solo_digitos(t, nombre_campo, min_len, max_len)


def v_correo(t, v):
    """Campo de correo electrónico."""
    return validar_correo(t)


def v_entero(nombre_campo, minimo=None, maximo=None):
    """Campo numérico entero dentro de un rango."""
    return lambda t, v: validar_entero(t, nombre_campo, minimo, maximo)


def v_fecha(minimo=None, maximo=None):
    """Campo de fecha AAAA-MM-DD entre minimo y maximo (date, texto o función(valores))."""
    def validar(t, v):
        f = a_fecha(validar_fecha(t, 'fecha'))
        lo, hi = resolver(minimo, v), resolver(maximo, v)
        if lo and f < a_fecha(validar_fecha(lo, 'mínimo')):
            raise ValueError(f'La fecha no puede ser anterior a {lo}')
        if hi and f > a_fecha(validar_fecha(hi, 'máximo')):
            raise ValueError(f'La fecha no puede ser posterior a {hi}')
        return f
    return validar


def v_documento_nuevo(t, v):
    """Documento válido que aún no esté registrado."""
    doc = validar_solo_digitos(t, 'número de documento', 5, 12)
    if repo_usuario_find_by_documento(doc):
        raise ValueError(f'Ya existe un usuario con documento {doc}')
    return doc


def v_documento_existente(t, v):
    """Documento de un usuario registrado (muestra su nombre)."""
    u = usuario_buscar_por_documento(validar_solo_digitos(t, 'número de documento', 5, 12))
    print(f'      → {u["nombre"]} ({u["tipo"]}, {u["estado"]})')
    return u['num_documento']


def v_documento_habilitado(t, v):
    """Usuario registrado, ACTIVO y sin multas pendientes."""
    doc = v_documento_existente(t, v)
    if repo_usuario_find_by_documento(doc)['estado'] != 'ACTIVO':
        raise ValueError('El usuario está SUSPENDIDO')
    pendientes = multa_listar_pendientes_usuario(doc)
    if pendientes:
        raise ValueError(f'El usuario tiene {len(pendientes)} multa(s) pendiente(s) por '
                         f'${sum(m["valor"] for m in pendientes):,} COP')
    return doc


def v_isbn(t, v):
    """ISBN con formato válido (10 o 13 dígitos)."""
    isbn = validar_solo_digitos(t.replace('-', ''), 'ISBN', 10, 13)
    if len(isbn) not in (10, 13):
        raise ValueError(f'El ISBN debe tener 10 o 13 dígitos (tiene {len(isbn)})')
    return isbn


def v_isbn_nuevo(t, v):
    """ISBN válido que aún no esté registrado."""
    isbn = v_isbn(t, v)
    if repo_libro_find_by_isbn(isbn):
        raise ValueError(f'Ya existe un libro con ISBN {isbn}')
    return isbn


def v_isbn_existente(t, v):
    """ISBN de un libro registrado (muestra título y disponibilidad)."""
    l = libro_buscar_por_isbn(v_isbn(t, v))
    print(f'      → "{l["titulo"]}" ({l["disponibles"]}/{l["total_ejemplares"]} disponibles)')
    return l['isbn']


def v_codigo_categoria_nuevo(t, v):
    """Código de categoría con formato válido y no registrado."""
    cod = crear_categoria(t, 'Temporal')['codigo']
    if repo_categoria_find_by_codigo(cod):
        raise ValueError(f'Ya existe la categoría {cod}')
    return cod


def v_codigo(prefijo, buscar, condicion=None, mensaje=''):
    """Código PRE-/MUL-/RES- seguido de 17 dígitos que exista (y cumpla la condición)."""
    def validar(t, v):
        cod = t.strip().upper()
        if not re.fullmatch(prefijo + r'-\d{17}', cod):
            raise ValueError(f'El código debe tener el formato {prefijo}- seguido de 17 dígitos')
        registro = buscar(cod)
        if registro is None:
            raise ValueError(f'No existe el código {cod}')
        if condicion and not condicion(registro):
            raise ValueError(mensaje)
        return cod
    return validar


def opciones_enum(enum):
    """Lista de opciones (valor, texto) a partir de un enum."""
    return [(x, x) for x in enum]


def opciones_categorias(valores=None):
    """Lista de opciones con las categorías registradas."""
    return [(c['codigo'], f'{c["codigo"]} — {c["nombre"]}') for c in libro_listar_categorias()]


# ---------- Presentación ----------

def leer_opcion(titulo, opciones):
    """Imprime un menú (dict opción -> texto) y retorna la opción elegida."""
    print(f'\n===== {titulo} =====')
    for clave, texto in opciones.items():
        print(f' {clave}. {texto}')
    return leer_texto('Opción: ')


def imprimir_tabla(encabezados, filas):
    """Imprime una tabla de texto con columnas alineadas."""
    if not filas:
        print('  (sin registros)')
        return
    filas = [['' if v is None else str(v) for v in f] for f in filas]
    anchos = [max(len(str(e)), *(len(f[i]) for f in filas)) for i, e in enumerate(encabezados)]
    print('  ' + ' | '.join(str(e).ljust(a) for e, a in zip(encabezados, anchos)))
    print('  ' + '-+-'.join('-' * a for a in anchos))
    for f in filas:
        print('  ' + ' | '.join(v.ljust(a) for v, a in zip(f, anchos)))


def mostrar_usuarios(usuarios):
    """Imprime una lista de usuarios."""
    imprimir_tabla(['Documento', 'Nombre', 'Tipo', 'Correo', 'Teléfono', 'Estado'],
                   [[u['num_documento'], u['nombre'], u['tipo'], u['correo'], u['telefono'], u['estado']]
                    for u in usuarios])


def mostrar_libros(libros):
    """Imprime una lista de libros."""
    imprimir_tabla(['ISBN', 'Título', 'Autor', 'Cat.', 'Año', 'Total', 'Disp.'],
                   [[l['isbn'], l['titulo'], l['autor'], l['cod_categoria'], l['anio'],
                     l['total_ejemplares'], l['disponibles']] for l in libros])


def mostrar_prestamos(prestamos):
    """Imprime una lista de préstamos."""
    imprimir_tabla(['Código', 'ISBN', 'Documento', 'Préstamo', 'Límite', 'Devolución', 'Estado'],
                   [[p['codigo'], p['isbn'], p['num_documento'], p['fecha_prestamo'], p['fecha_limite'],
                     p['fecha_devolucion'] or VACIO, p['estado']] for p in prestamos])


def mostrar_multas(multas):
    """Imprime una lista de multas."""
    imprimir_tabla(['Código', 'Préstamo', 'Días', 'Valor', 'Estado'],
                   [[m['codigo'], m['cod_prestamo'], m['dias_retraso'], f"${m['valor']:,}", m['estado']]
                    for m in multas])


def mostrar_reservas(reservas):
    """Imprime una lista de reservas."""
    imprimir_tabla(['Código', 'ISBN', 'Documento', 'Fecha', 'Estado'],
                   [[r['codigo'], r['isbn'], r['num_documento'], r['fecha_reserva'], r['estado']]
                    for r in reservas])


def ejecutar_menu(titulo, acciones):
    """Bucle genérico de menú: acciones es un dict opción -> (texto, función). '0' vuelve."""
    opciones = {k: texto for k, (texto, _) in acciones.items()}
    opciones['0'] = 'Volver' if titulo != 'SGB — MENÚ PRINCIPAL' else 'Salir'
    while True:
        op = leer_opcion(titulo, opciones)
        if op == '0':
            return
        if op not in acciones:
            print(f'  Opción no válida. Escriba un número de 0 a {len(acciones)}.')
            continue
        try:
            acciones[op][1]()
        except ValueError as e:
            print(f'  ERROR: {e}')

## Celda 4.1 — menu_usuarios()

In [ ]:
def ui_registrar_usuario():
    """Formulario de registro de usuario."""
    d = leer_formulario('Registrar usuario', [
        campo('doc', 'Número de documento (5-12 dígitos)', v_documento_nuevo),
        campo('nombre', 'Nombre completo (solo letras)', v_letras('nombre')),
        campo('tipo', 'Tipo de usuario', opciones=opciones_enum(TIPO_USUARIO_ENUM)),
        campo('correo', 'Correo', v_correo),
        campo('tel', 'Teléfono (7-10 dígitos)', v_digitos('teléfono', 7, 10)),
    ])
    if d:
        u = usuario_registrar(d['doc'], d['nombre'], d['tipo'], d['correo'], d['tel'])
        print(f'  Usuario {u["nombre"]} registrado.')


def ui_consultar_usuario():
    """Muestra un usuario, sus préstamos abiertos y sus multas pendientes."""
    d = leer_formulario('Consultar usuario', [campo('doc', 'Número de documento', v_documento_existente)],
                        confirmar=False)
    if not d:
        return
    u = usuario_buscar_por_documento(d['doc'])
    mostrar_usuarios([u])
    reglas = REGLAS_POR_TIPO[u['tipo']]
    abiertos = [p for p in prestamo_listar_por_usuario(u['num_documento']) if p['estado'] != 'DEVUELTO']
    print(f'  Préstamos abiertos: {len(abiertos)}/{reglas["cupo"]} · plazo {reglas["dias"]} días')
    pendientes = multa_listar_pendientes_usuario(u['num_documento'])
    print(f'  Multas pendientes: {len(pendientes)} (${sum(m["valor"] for m in pendientes):,})')


def ui_actualizar_contacto():
    """Actualiza nombre, correo y teléfono (Enter conserva el valor actual)."""
    actual = lambda v, k: repo_usuario_find_by_documento(v['doc'])[k]
    d = leer_formulario('Actualizar datos de contacto  (Enter conserva el valor actual)', [
        campo('doc', 'Número de documento', v_documento_existente),
        campo('nombre', 'Nombre completo', v_letras('nombre'), por_defecto=lambda v: actual(v, 'nombre')),
        campo('correo', 'Correo', v_correo, por_defecto=lambda v: actual(v, 'correo')),
        campo('tel', 'Teléfono', v_digitos('teléfono', 7, 10), por_defecto=lambda v: actual(v, 'telefono')),
    ])
    if d:
        mostrar_usuarios([usuario_actualizar(d['doc'], nombre=d['nombre'], correo=d['correo'],
                                             telefono=d['tel'])])


def ui_cambiar_estado():
    """Activa o suspende un usuario."""
    d = leer_formulario('Activar / suspender usuario', [
        campo('doc', 'Número de documento', v_documento_existente),
        campo('estado', 'Nuevo estado', opciones=opciones_enum(ESTADO_USUARIO_ENUM)),
    ])
    if d:
        u = usuario_actualizar(d['doc'], estado=d['estado'])
        print(f'  {u["nombre"]} ahora está {u["estado"]}.')


def menu_usuarios():
    """Menú de gestión de usuarios (RF-01)."""
    ejecutar_menu('GESTIÓN DE USUARIOS', {
        '1': ('Registrar usuario', ui_registrar_usuario),
        '2': ('Consultar usuario por documento', ui_consultar_usuario),
        '3': ('Actualizar datos de contacto', ui_actualizar_contacto),
        '4': ('Activar / suspender usuario', ui_cambiar_estado),
        '5': ('Listar todos los usuarios', lambda: mostrar_usuarios(usuario_listar_todos())),
    })

## Celda 4.2 — menu_libros()

In [ ]:
def ui_registrar_categoria():
    """Formulario de registro de categoría."""
    d = leer_formulario('Registrar categoría', [
        campo('cod', 'Código (2-5 letras, ej. LIT)', v_codigo_categoria_nuevo),
        campo('nombre', 'Nombre (solo letras)', v_letras('nombre de categoría', 40)),
        campo('desc', 'Descripción', v_texto('descripción'), opcional=True),
    ])
    if d:
        c = libro_registrar_categoria(d['cod'], d['nombre'], d['desc'])
        print(f'  Categoría {c["codigo"]} registrada.')


def ui_listar_categorias():
    """Muestra las categorías."""
    imprimir_tabla(['Código', 'Nombre', 'Descripción'],
                   [[c['codigo'], c['nombre'], c['descripcion'] or ''] for c in libro_listar_categorias()])


def ui_registrar_libro():
    """Formulario de registro de libro."""
    d = leer_formulario('Registrar libro', [
        campo('isbn', 'ISBN (10 o 13 dígitos)', v_isbn_nuevo),
        campo('titulo', 'Título', v_texto('título')),
        campo('autor', 'Autor (solo letras)', v_letras('autor')),
        campo('cat', 'Categoría', opciones=opciones_categorias),
        campo('anio', f'Año de publicación (1000-{date.today().year})',
              v_entero('año', 1000, date.today().year)),
        campo('total', 'Número de ejemplares (1-999)', v_entero('ejemplares', 1, 999)),
    ])
    if d:
        l = libro_registrar(d['isbn'], d['titulo'], d['autor'], d['cat'], d['anio'], d['total'])
        print(f'  Libro "{l["titulo"]}" registrado con {l["total_ejemplares"]} ejemplar(es).')


def ui_consultar_isbn():
    """Consulta un libro por ISBN."""
    d = leer_formulario('Consultar libro', [campo('isbn', 'ISBN', v_isbn_existente)], confirmar=False)
    if d:
        mostrar_libros([libro_buscar_por_isbn(d['isbn'])])


def ui_buscar_titulo():
    """Busca libros por parte del título."""
    d = leer_formulario('Buscar por título', [campo('txt', 'Texto del título', v_texto('título'))],
                        confirmar=False)
    if d:
        mostrar_libros(libro_buscar_por_titulo(d['txt']))


def ui_listar_por_categoria():
    """Lista los libros de una categoría elegida de la lista."""
    d = leer_formulario('Libros por categoría', [campo('cat', 'Categoría', opciones=opciones_categorias)],
                        confirmar=False)
    if d:
        mostrar_libros(libro_listar_por_categoria(d['cat']))


def menu_libros():
    """Menú de gestión de libros y categorías (RF-02)."""
    ejecutar_menu('LIBROS Y CATEGORÍAS', {
        '1': ('Registrar categoría', ui_registrar_categoria),
        '2': ('Listar categorías', ui_listar_categorias),
        '3': ('Registrar libro', ui_registrar_libro),
        '4': ('Consultar libro por ISBN', ui_consultar_isbn),
        '5': ('Buscar libros por título', ui_buscar_titulo),
        '6': ('Listar libros por categoría', ui_listar_por_categoria),
        '7': ('Listar libros con ejemplares disponibles', lambda: mostrar_libros(libro_listar_disponibles())),
    })

## Celda 4.3 — menu_prestamos()

In [ ]:
def ui_prestar():
    """Formulario de préstamo."""
    d = leer_formulario('Registrar préstamo', [
        campo('doc', 'Documento del usuario', v_documento_habilitado),
        campo('isbn', 'ISBN del libro', v_isbn_existente),
        campo('fecha', 'Fecha del préstamo (AAAA-MM-DD)', v_fecha(maximo=date.today()),
              por_defecto=date.today()),
    ])
    if d:
        p = prestamo_prestar(d['doc'], d['isbn'], d['fecha'])
        print(f'  Préstamo {p["codigo"]} registrado. Fecha límite: {p["fecha_limite"]}')


def ui_devolver():
    """Formulario de devolución; informa la multa si la hay."""
    abierto = lambda p: p['estado'] != 'DEVUELTO'
    d = leer_formulario('Registrar devolución', [
        campo('cod', 'Código del préstamo (PRE-...)',
              v_codigo('PRE', repo_prestamo_find_by_codigo, abierto, 'Ese préstamo ya fue devuelto')),
        campo('fecha', 'Fecha de devolución (AAAA-MM-DD)',
              v_fecha(minimo=lambda v: repo_prestamo_find_by_codigo(v['cod'])['fecha_prestamo'],
                      maximo=date.today()), por_defecto=date.today()),
    ])
    if not d:
        return
    p, multa = prestamo_devolver(d['cod'], d['fecha'])
    print(f'  Devolución de {p["codigo"]} registrada.')
    if multa:
        print(f'  Entregado con {multa["dias_retraso"]} día(s) de retraso: multa {multa["codigo"]} '
              f'por ${multa["valor"]:,} COP')
    else:
        print('  Entregado a tiempo, sin multa.')


def ui_consultar_prestamo():
    """Consulta un préstamo por código."""
    d = leer_formulario('Consultar préstamo', [
        campo('cod', 'Código del préstamo (PRE-...)', v_codigo('PRE', repo_prestamo_find_by_codigo))],
        confirmar=False)
    if d:
        mostrar_prestamos([prestamo_buscar_por_codigo(d['cod'])])


def ui_prestamos_usuario():
    """Lista los préstamos de un usuario."""
    d = leer_formulario('Préstamos de un usuario', [campo('doc', 'Documento', v_documento_existente)],
                        confirmar=False)
    if d:
        mostrar_prestamos(prestamo_listar_por_usuario(d['doc']))


def ui_vencidos():
    """Lista los préstamos vencidos a una fecha de corte."""
    d = leer_formulario('Préstamos vencidos', [
        campo('fecha', 'Fecha de corte (AAAA-MM-DD)', v_fecha(), por_defecto=date.today())], confirmar=False)
    if d:
        mostrar_prestamos(prestamo_listar_vencidos(d['fecha']))


def menu_prestamos():
    """Menú de gestión de préstamos (RF-03)."""
    ejecutar_menu('PRÉSTAMOS', {
        '1': ('Registrar préstamo', ui_prestar),
        '2': ('Registrar devolución', ui_devolver),
        '3': ('Consultar préstamo por código', ui_consultar_prestamo),
        '4': ('Listar préstamos de un usuario', ui_prestamos_usuario),
        '5': ('Listar préstamos activos', lambda: mostrar_prestamos(prestamo_listar_activos())),
        '6': ('Listar préstamos vencidos a una fecha', ui_vencidos),
    })

## Celda 4.4 — menu_reservas_multas()

In [ ]:
def ui_reservar():
    """Formulario de reserva."""
    d = leer_formulario('Reservar libro', [
        campo('doc', 'Documento del usuario', v_documento_existente),
        campo('isbn', 'ISBN del libro', v_isbn_existente),
    ])
    if d:
        r = reserva_reservar(d['doc'], d['isbn'])
        print(f'  Reserva {r["codigo"]} creada. Puesto en la cola: {len(reserva_listar_por_libro(r["isbn"]))}')


def ui_cancelar_reserva():
    """Cancela una reserva activa."""
    d = leer_formulario('Cancelar reserva', [
        campo('cod', 'Código de la reserva (RES-...)',
              v_codigo('RES', repo_reserva_find_by_codigo, lambda r: r['estado'] == 'ACTIVA',
                       'Esa reserva no está ACTIVA'))])
    if d:
        print(f'  Reserva {reserva_cancelar(d["cod"])["codigo"]} cancelada.')


def ui_cola_reservas():
    """Muestra la cola de reservas de un libro."""
    d = leer_formulario('Cola de reservas', [campo('isbn', 'ISBN', v_isbn_existente)], confirmar=False)
    if d:
        mostrar_reservas(reserva_listar_por_libro(d['isbn']))


def ui_multas_pendientes():
    """Muestra las multas pendientes de un usuario."""
    d = leer_formulario('Multas pendientes', [campo('doc', 'Documento del usuario', v_documento_existente)],
                        confirmar=False)
    if d:
        pendientes = multa_listar_pendientes_usuario(d['doc'])
        mostrar_multas(pendientes)
        print(f'  Total pendiente: ${sum(m["valor"] for m in pendientes):,} COP')


def ui_pagar_multa():
    """Registra el pago de una multa."""
    d = leer_formulario('Pagar multa', [
        campo('cod', 'Código de la multa (MUL-...)',
              v_codigo('MUL', repo_multa_find_by_codigo, lambda m: m['estado'] == 'PENDIENTE',
                       'Esa multa ya está PAGADA'))])
    if d:
        m = multa_pagar(d['cod'])
        print(f'  Multa {m["codigo"]} por ${m["valor"]:,} COP pagada.')


def ui_historial_usuario():
    """Historial cronológico de préstamos, multas y reservas de un usuario."""
    d = leer_formulario('Historial de usuario', [campo('doc', 'Documento del usuario', v_documento_existente)],
                        confirmar=False)
    if not d:
        return
    u = usuario_buscar_por_documento(d['doc'])
    print(f'\n  HISTORIAL DE {u["nombre"].upper()} ({u["tipo"]}, {u["estado"]})')
    titulos = {l['isbn']: l['titulo'] for l in repo_libro_find_all()}
    imprimir_tabla(['Código', 'Libro', 'Préstamo', 'Límite', 'Devolución', 'Estado'],
                   [[p['codigo'], titulos.get(p['isbn'], p['isbn']), p['fecha_prestamo'], p['fecha_limite'],
                     p['fecha_devolucion'] or VACIO, p['estado']]
                    for p in prestamo_historial_usuario(u['num_documento'])])
    print('\n  Multas:')
    mostrar_multas(multa_listar_por_usuario(u['num_documento']))
    print('\n  Reservas:')
    mostrar_reservas(reserva_listar_por_usuario(u['num_documento']))


def menu_reservas_multas():
    """Menú de reservas, multas e historial (RF-04)."""
    ejecutar_menu('RESERVAS, MULTAS E HISTORIAL', {
        '1': ('Reservar libro', ui_reservar),
        '2': ('Cancelar reserva', ui_cancelar_reserva),
        '3': ('Ver cola de reservas de un libro', ui_cola_reservas),
        '4': ('Consultar multas pendientes de un usuario', ui_multas_pendientes),
        '5': ('Pagar multa', ui_pagar_multa),
        '6': ('Historial de un usuario', ui_historial_usuario),
    })

## Celda 4.5 — menu_analisis()

In [ ]:
def ui_libros_mas_prestados():
    """Top N de libros más prestados en un periodo."""
    hoy = date.today()
    d = leer_formulario('Libros más prestados', [
        campo('ini', 'Fecha inicio (AAAA-MM-DD)', v_fecha(), por_defecto=hoy - timedelta(days=365)),
        campo('fin', 'Fecha fin (AAAA-MM-DD)', v_fecha(minimo=lambda v: v['ini']), por_defecto=hoy),
        campo('n', 'Cantidad de libros a mostrar (1-50)', v_entero('cantidad', 1, 50), por_defecto=10),
    ], confirmar=False)
    if d:
        imprimir_tabla(['#', 'ISBN', 'Título', 'Préstamos'],
                       [[i, isbn, t, c] for i, (isbn, t, c)
                        in enumerate(analisis_libros_mas_prestados(d['ini'], d['fin'], d['n']), 1)])


def ui_prestamos_por_categoria():
    """Distribución de préstamos por categoría."""
    imprimir_tabla(['Código', 'Categoría', 'Préstamos', '%'],
                   [[c, n, k, f'{p}%'] for c, n, k, p in analisis_prestamos_por_categoria()])


def ui_morosidad():
    """Tasa de morosidad por tipo de usuario."""
    imprimir_tabla(['Tipo de usuario', 'Total préstamos', 'Préstamos morosos', 'Tasa (%)'],
                   [[t, tot, mor, f'{tasa}%'] for t, (tot, mor, tasa) in analisis_morosidad_por_tipo().items()])


def ui_duracion_promedio():
    """Duración promedio de los préstamos devueltos."""
    prom = analisis_duracion_promedio_prestamo()
    print('  No hay préstamos devueltos.' if prom is None else f'  Duración promedio: {prom} días')


def ui_reporte_multas():
    """Reporte financiero de multas."""
    r = analisis_reporte_multas()
    imprimir_tabla(['Concepto', 'Cantidad', 'Valor (COP)'], [
        ['Generado', r['cantidad'], f"${r['generado']:,}"],
        ['Pagado', r['cant_pagadas'], f"${r['pagado']:,}"],
        ['Pendiente', r['cant_pendientes'], f"${r['pendiente']:,}"],
    ])


def menu_analisis():
    """Menú del módulo de análisis de datos (RF-05)."""
    ejecutar_menu('ANÁLISIS DE DATOS', {
        '1': ('Listar los N libros más prestados en un periodo', ui_libros_mas_prestados),
        '2': ('Ver distribución de préstamos por categoría', ui_prestamos_por_categoria),
        '3': ('Ver tasa de morosidad por tipo de usuario', ui_morosidad),
        '4': ('Ver duración promedio de los préstamos devueltos', ui_duracion_promedio),
        '5': ('Ver reporte de multas', ui_reporte_multas),
    })

## Celda 4.6 — menu_principal()

In [ ]:
def menu_principal():
    """Menú principal del sistema: opciones 1-5 y 0 para salir."""
    limpiar_pantalla()
    prestamo_actualizar_vencidos()
    ejecutar_menu('SGB — MENÚ PRINCIPAL', {
        '1': ('Gestión de usuarios', menu_usuarios),
        '2': ('Gestión de libros y categorías', menu_libros),
        '3': ('Gestión de préstamos', menu_prestamos),
        '4': ('Reservas, multas e historial', menu_reservas_multas),
        '5': ('Análisis de datos', menu_analisis),
    })
    print('Hasta pronto.')

---
# SECCIÓN 5 — EJECUCIÓN
## Celda 5.0 — Datos de prueba y pruebas de integración
* `cargar_datos_prueba()` llena la carpeta de datos **solo si está vacía** (fechas relativas a hoy, para que
  siempre haya préstamos activos, vencidos y devueltos).
* `pruebas_integracion()` ejecuta el flujo completo en una **carpeta temporal** (no modifica los datos reales).

In [ ]:
def cargar_datos_prueba(forzar=False):
    """Escribe datos de ejemplo en la carpeta de datos si está vacía (o si forzar=True)."""
    if not forzar and (leer_lineas(RUTA_USUARIOS) or leer_lineas(RUTA_LIBROS)):
        return False
    hoy = date.today()

    def dia(n):
        """Fecha relativa a hoy."""
        return hoy + timedelta(days=n)

    def cod(prefijo, n, i):
        """Código con el formato de generar_codigo para una fecha relativa."""
        return f'{prefijo}-{dia(n):%Y%m%d}090000{i:03d}'

    categorias = [crear_categoria('LIT', 'Literatura', 'Novela, cuento y poesía'),
                  crear_categoria('TEC', 'Tecnología', 'Programación e ingeniería'),
                  crear_categoria('CIE', 'Ciencias', 'Ciencias naturales y divulgación'),
                  crear_categoria('HIS', 'Historia', 'Historia de Colombia y del mundo')]
    usuarios = [crear_usuario('1098765432', 'María López', 'ESTUDIANTE', 'maria.lopez@correo.com', '3001234567'),
                crear_usuario('1001234567', 'Juan Pérez', 'DOCENTE', 'juan.perez@correo.com', '3109876543'),
                crear_usuario('1005556677', 'Carlos Ruiz', 'ESTUDIANTE', 'carlos.ruiz@correo.com', '3204567890'),
                crear_usuario('79888777', 'Ana Torres', 'EXTERNO', 'ana.torres@correo.com', '3157778899'),
                crear_usuario('1102223344', 'Laura Gómez', 'ESTUDIANTE', 'laura.gomez@correo.com', '3012223344',
                              'SUSPENDIDO')]
    # (isbn, titulo, autor, categoria, anio, total, disponibles)
    libros = [crear_libro('9789585001234', 'Cien años de soledad', 'Gabriel García Márquez', 'LIT', 1967, 5, 4),
              crear_libro('9789587654321', 'Estructuras de datos en Python', 'Ana Gómez', 'TEC', 2020, 2, 1),
              crear_libro('9780132350884', 'Clean Code', 'Robert C. Martin', 'TEC', 2008, 3, 3),
              crear_libro('9789584295670', 'Breve historia de Colombia', 'Jorge Orlando Melo', 'HIS', 2017, 2, 1),
              crear_libro('9788408106463', 'Cosmos', 'Carl Sagan', 'CIE', 1980, 2, 2),
              crear_libro('9789588886095', 'El coronel no tiene quien le escriba', 'Gabriel García Márquez',
                          'LIT', 1961, 1, 0)]
    # (codigo, isbn, documento, prestamo, limite, devolucion, estado)
    prestamos = [
        crear_prestamo(cod('PRE', -60, 1), '9789585001234', '1098765432', dia(-60), dia(-53), dia(-50), 'DEVUELTO'),
        crear_prestamo(cod('PRE', -45, 2), '9780132350884', '1001234567', dia(-45), dia(-30), dia(-35), 'DEVUELTO'),
        crear_prestamo(cod('PRE', -40, 3), '9789587654321', '1005556677', dia(-40), dia(-33), dia(-30), 'DEVUELTO'),
        crear_prestamo(cod('PRE', -30, 4), '9788408106463', '79888777', dia(-30), dia(-25), dia(-20), 'DEVUELTO'),
        crear_prestamo(cod('PRE', -25, 5), '9780132350884', '79888777', dia(-25), dia(-20), dia(-22), 'DEVUELTO'),
        crear_prestamo(cod('PRE', -20, 6), '9789585001234', '1005556677', dia(-20), dia(-13), dia(-14), 'DEVUELTO'),
        crear_prestamo(cod('PRE', -12, 7), '9789587654321', '1098765432', dia(-12), dia(-5), None, 'VENCIDO'),
        crear_prestamo(cod('PRE', -5, 8), '9789585001234', '1001234567', dia(-5), dia(10), None, 'ACTIVO'),
        crear_prestamo(cod('PRE', -3, 9), '9789588886095', '1001234567', dia(-3), dia(12), None, 'ACTIVO'),
        crear_prestamo(cod('PRE', -2, 10), '9789584295670', '1098765432', dia(-2), dia(5), None, 'ACTIVO'),
    ]
    multas = [crear_multa(cod('MUL', -50, 1), prestamos[0]['codigo'], 3, 3000, 'PAGADA'),
              crear_multa(cod('MUL', -30, 2), prestamos[2]['codigo'], 3, 3000, 'PENDIENTE'),
              crear_multa(cod('MUL', -20, 3), prestamos[3]['codigo'], 5, 5000, 'PAGADA')]
    reservas = [crear_reserva(cod('RES', -8, 1), '9789588886095', '1001234567', dia(-8), 'CUMPLIDA'),
                crear_reserva(cod('RES', -10, 2), '9789587654321', '1005556677', dia(-10), 'CANCELADA'),
                crear_reserva(cod('RES', -1, 3), '9789588886095', '79888777', dia(-1), 'ACTIVA')]

    reescribir_registros(RUTA_CATEGORIAS, categorias, categoria_to_linea)
    reescribir_registros(RUTA_USUARIOS, usuarios, usuario_to_linea)
    reescribir_registros(RUTA_LIBROS, libros, libro_to_linea)
    reescribir_registros(RUTA_PRESTAMOS, prestamos, prestamo_to_linea)
    reescribir_registros(RUTA_MULTAS, multas, multa_to_linea)
    reescribir_registros(RUTA_RESERVAS, reservas, reserva_to_linea)
    return True


def esperar_error(funcion, *args):
    """Verifica que la función lance ValueError (usada en las pruebas)."""
    try:
        funcion(*args)
    except ValueError:
        return
    raise AssertionError(f'{funcion.__name__}{args} debió lanzar ValueError')


def pruebas_integracion():
    """Flujo completo en una carpeta temporal: registro, préstamo, vencimiento, multa, pago,
    reservas por orden de llegada e indicadores de análisis."""
    import tempfile
    ruta_real = RUTA_BASE
    configurar_rutas(tempfile.mkdtemp())
    try:
        hoy = date.today()
        # RF-01 usuarios
        usuario_registrar('10000111', 'Ana Prueba', 'estudiante', 'ana@test.com', '3001110000')
        usuario_registrar('10000222', 'Beto Prueba', 'DOCENTE', 'beto@test.com', '3001110001')
        usuario_registrar('10000333', 'Caro Prueba', 'EXTERNO', 'caro@test.com', '3001110002')
        esperar_error(usuario_registrar, '10000111', 'Repetido', 'DOCENTE', 'r@test.com', '3001110003')
        esperar_error(usuario_registrar, '10000444', 'Mal tipo', 'ADMIN', 'm@test.com', '3001110004')
        esperar_error(usuario_registrar, '10000555', 'Con|pipe', 'DOCENTE', 'p@test.com', '3001110005')
        assert usuario_actualizar('10000111', correo='ana2@test.com')['correo'] == 'ana2@test.com'

        # RF-02 libros
        libro_registrar_categoria('LIT', 'Literatura')
        esperar_error(libro_registrar, '9780000000001', 'X', 'Y', 'NOEXISTE', 2000, 1)
        libro_registrar('9780000000001', 'Novela Uno', 'Autor A', 'LIT', 2001, 2)
        libro_registrar('9780000000002', 'Novela Dos', 'Autor B', 'lit', 2002, 1)
        assert len(libro_buscar_por_titulo('novela')) == 2
        assert len(libro_listar_por_categoria('LIT')) == 2

        # RF-03 préstamo atrasado -> vencido -> devolución -> multa (estudiante: 7 días)
        p1 = prestamo_prestar('10000111', '9780000000002', hoy - timedelta(days=10))
        assert p1['fecha_limite'] == str(hoy - timedelta(days=3))
        assert libro_buscar_por_isbn('9780000000002')['disponibles'] == 0
        assert [p['codigo'] for p in prestamo_listar_vencidos()] == [p1['codigo']]
        assert prestamo_buscar_por_codigo(p1['codigo'])['estado'] == 'VENCIDO'
        _, multa = prestamo_devolver(p1['codigo'], hoy)
        assert multa['dias_retraso'] == 3 and multa['valor'] == 3000
        esperar_error(prestamo_devolver, p1['codigo'])           # ya devuelto
        esperar_error(prestamo_prestar, '10000111', '9780000000001')  # multa pendiente
        multa_pagar(multa['codigo'])
        assert not multa_listar_pendientes_usuario('10000111')

        # Cupo: externo = 2 libros
        prestamo_prestar('10000333', '9780000000001')
        prestamo_prestar('10000333', '9780000000002')
        assert libro_buscar_por_isbn('9780000000002')['disponibles'] == 0

        # RF-04 reservas por orden de llegada
        reserva_reservar('10000222', '9780000000002', hoy - timedelta(days=1))
        reserva_reservar('10000111', '9780000000002', hoy)
        esperar_error(reserva_reservar, '10000222', '9780000000002')  # reserva duplicada
        p3 = [p for p in prestamo_listar_activos() if p['num_documento'] == '10000333'
              and p['isbn'] == '9780000000002'][0]
        prestamo_devolver(p3['codigo'])
        esperar_error(prestamo_prestar, '10000111', '9780000000002')  # Beto llegó primero
        prestamo_prestar('10000222', '9780000000002')                  # la reserva queda CUMPLIDA
        assert [r['estado'] for r in reserva_listar_por_usuario('10000222')] == ['CUMPLIDA']
        r_ana = reserva_listar_por_libro('9780000000002')[0]
        reserva_cancelar(r_ana['codigo'])
        esperar_error(reserva_cancelar, r_ana['codigo'])

        # Usuario suspendido no presta
        usuario_actualizar('10000111', estado='SUSPENDIDO')
        esperar_error(prestamo_prestar, '10000111', '9780000000001')

        # RF-05 análisis
        assert analisis_libros_mas_prestados(hoy - timedelta(days=30), hoy, 1)[0][2] == 3
        assert analisis_prestamos_por_categoria()[0][3] == 100.0
        assert analisis_morosidad_por_tipo()['ESTUDIANTE'] == (1, 1, 100.0)
        assert analisis_duracion_promedio_prestamo() is not None
        assert analisis_reporte_multas()['pagado'] == 3000
        assert len(prestamo_historial_usuario('10000111')) == 1

        # Validaciones de campos
        esperar_error(usuario_registrar, '12ab5678', 'Letras Doc', 'DOCENTE', 'x@test.com', '3001112233')
        esperar_error(usuario_registrar, '123', 'Doc Corto', 'DOCENTE', 'x@test.com', '3001112233')
        esperar_error(usuario_registrar, '10000666', 'Ana 2', 'DOCENTE', 'x@test.com', '3001112233')
        esperar_error(usuario_registrar, '10000666', 'Ana', 'DOCENTE', 'correo-malo', '3001112233')
        esperar_error(usuario_registrar, '10000666', 'Ana', 'DOCENTE', 'x@test.com', '300-ABC')
        esperar_error(libro_registrar_categoria, 'L1', 'Literatura')
        esperar_error(crear_libro, '97800000000', 'T', 'Autor', 'LIT', 2000, 1)       # 11 dígitos
        esperar_error(crear_libro, '9780000000009', 'T', 'Autor 3', 'LIT', 2000, 1)   # autor con número
        esperar_error(crear_libro, '9780000000009', 'T', 'Autor', 'LIT', 'dos mil', 1)
        esperar_error(crear_libro, '9780000000009', 'T', 'Autor', 'LIT', 2000, 0)
        esperar_error(prestamo_prestar, '10000111', '9780000000001', hoy + timedelta(days=1))

        # Datos de prueba consistentes
        configurar_rutas(tempfile.mkdtemp())
        assert cargar_datos_prueba()
        for l in repo_libro_find_all():
            abiertos = [p for p in repo_prestamo_find_by_libro(l['isbn']) if p['estado'] != 'DEVUELTO']
            assert l['disponibles'] == l['total_ejemplares'] - len(abiertos), l['titulo']
        print('Pruebas de integración: OK')
    finally:
        configurar_rutas(ruta_real)


pruebas_integracion()
if cargar_datos_prueba():
    print('Datos de prueba cargados en', RUTA_BASE)
else:
    print('Se usan los datos existentes en', RUTA_BASE)

## Celda 5.1 — Ejecutar para iniciar

In [ ]:
menu_principal()